# Quality Control
##### Franziska Niemeyer, 2026-03-16

In [ ]:
import numpy
import os
import scanpy as sc
import numpy as np
import sys
import math
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

sc.set_figure_params(color_map='viridis_r', dpi_save=600, vector_friendly=True, fontsize=12)
color_palette = "Set1"

In [ ]:
WORKING_DIR = "."
DATA_DIR = "data"
OUT_DIR = "figures"
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

sc.settings.figdir = OUT_DIR

##### Load data

In [ ]:
aggr_matrix = os.path.join(DATA_DIR, "outs/filtered_feature_bc_matrix.h5")
adata = sc.read_10x_h5(aggr_matrix)
adata.var_names_make_unique()
adata

##### Load Spatial Data

In [ ]:
import scanpy as sc
import squidpy as sq
from PIL import Image
import os
import json

def load_aggregated_visium(
        aggr_path: str,
        barcode_suffix_to_sample: dict,
        spatial_base: str = None,
        library_id: str = None
):
    """
    Load aggregated Visium data and map sample names using barcode suffixes,
    then attach spatial images per sample using Squidpy.
    """
    adata = sc.read_visium(aggr_path, load_images=False)
    adata.var_names_make_unique()

    barcodes = adata.obs_names
    suffixes = [bc.split('-')[-1] for bc in barcodes]

    sample_names = []
    for suffix in suffixes:
        if suffix not in barcode_suffix_to_sample:
            raise ValueError(f"Suffix '{suffix}' not found in mapping.")
        sample_names.append(barcode_suffix_to_sample[suffix])

    adata.obs['sample'] = sample_names

    if spatial_base is None:
        spatial_base = os.path.join(aggr_path, "spatial")

    for sample in set(sample_names):
        sample_spatial_path = os.path.join(spatial_base, 'spatial', sample)
        tissue_img_path = os.path.join(sample_spatial_path, "tissue_hires_image.png")
        scalefactors_path = os.path.join(sample_spatial_path, "scalefactors_json.json")

        if not os.path.isfile(tissue_img_path) or not os.path.isfile(scalefactors_path):
            raise FileNotFoundError(f"Missing spatial image or scalefactors for {sample}")

        img = Image.open(tissue_img_path)
        img_np = np.array(img)

        with open(scalefactors_path) as f:
            scale_factors = json.load(f)

        adata.uns[f"spatial"][sample] = {
            "images": {"hires": img_np},
            "scalefactors": scale_factors,
        }

    spatial_df = pd.DataFrame(np.full((adata.n_obs, 2), np.nan),
                              index=adata.obs_names,
                              columns=["x", "y"])

    for suffix, sample in barcode_suffix_to_sample.items():

        # Load correct file
        for filename in ["tissue_positions.csv", "tissue_positions_list.csv", "aggr_tissue_positions.csv"]:
            positions_path = os.path.join(spatial_base, filename)
            if os.path.exists(positions_path):
                break
        else:
            raise FileNotFoundError(f"No tissue position file found for sample '{sample}'")

        coords = pd.read_csv(positions_path, header=None)
        coords.columns = ['barcode', 'in_tissue', 'array_row', 'array_col', 'pxl_row_in_fullres', 'pxl_col_in_fullres']

        if not filename in ["aggr_tissue_positions.csv", "tissue_positions_list.csv"]:
            # coords['barcode_full'] = coords['barcode'].astype(str) + f"-{suffix}"
            coords['barcode_full'] = coords['barcode'].map(lambda x: x.replace('-1', f'-{suffix}'))
            coords.set_index('barcode_full', inplace=True)
        else:
            coords.set_index('barcode', inplace=True)

        coords = coords.loc[coords.index.intersection(spatial_df.index)]
        spatial_df.loc[coords.index] = coords[["pxl_col_in_fullres", "pxl_row_in_fullres"]].astype(int).values  # x, y

    adata.obsm["spatial"] = spatial_df.to_numpy(dtype=int)

    return adata

In [ ]:
barcode_suffix_to_sample = {
    '1': 'Paxgene1',
    '2': 'Paxgene2',
    '3': 'Paxgene3',
    '4': 'Paxgene4'
}

adata = load_aggregated_visium(
    aggr_path=os.path.join(DATA_DIR, 'outs'),
    barcode_suffix_to_sample=barcode_suffix_to_sample,
    spatial_base=os.path.join(DATA_DIR, 'outs')
)

In [ ]:
anno_file = os.path.join(DATA_DIR, f"annotations/aggr/PAX_annotations.csv")
patients_file = os.path.join(DATA_DIR, f"annotations/aggr/PAX_patients.csv")
class_file = os.path.join(DATA_DIR, f"annotations/aggr/PAX_class.csv")

annos = lambda folder, name: pd.read_csv(folder, index_col=0, names=[name], header=0)
adata.obs = adata.obs.merge(how='left', right=annos(anno_file, 'histology'), left_index=True, right_index=True)
adata.obs = adata.obs.merge(how='left', right=annos(patients_file, 'patient'), left_index=True, right_index=True)
adata.obs = adata.obs.merge(how='left', right=annos(class_file, 'class'), left_index=True, right_index=True)

adata.obs['PFI'] = adata.obs['patient'].replace({'H110': 'long', 'H119': 'long', 'H145': 'short', 'H152': 'short', 'H178': 'short', 'H180': 'medium', 'H197': 'medium', 'H213': 'long', 'H233': 'short'})
adata.obs

#### Calculate QC metrics

In [ ]:
# No mitochondrial QC metrics -> v1 data doesn't contain MT gene probes
sc.pp.calculate_qc_metrics(
    adata,
    percent_top=None,  # Skip top expressed genes for now
    inplace=True       # Add results directly to adata.obs
)

print(adata.obs[['total_counts', 'n_genes_by_counts']].head())

In [ ]:
import seaborn as sns

adata.var_names_make_unique()
sns.jointplot(
    data=adata.obs,
    x="total_counts",
    y="n_genes_by_counts",
    kind="hex",
)
plt.savefig(os.path.join(OUT_DIR, "total_counts_vs_n_genes.pdf"))
plt.show()

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# Plot histograms for recalculated QC metrics
fig, axs = plt.subplots(1, 2, figsize=(18, 5))

# Total counts
sns.histplot(adata.obs["total_counts"], kde=True, bins=100, ax=axs[0])
axs[0].set_title("Total Counts")
axs[0].set_xlabel("Counts")
axs[0].set_ylabel("Frequency")

# Number of genes by counts
sns.histplot(adata.obs["n_genes_by_counts"], kde=True, bins=100, ax=axs[1])
axs[1].set_title("Number of Genes by Counts")
axs[1].set_xlabel("Number of Genes")
axs[1].set_ylabel("Frequency")

# Adjust layout for better visualization
plt.tight_layout()
plt.show()

#### Plot QC metrics spatially

In [ ]:
slides = adata.obs["sample"].unique()
n = len(slides)

ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes = axes.flatten()

for i, slide in enumerate(slides):
    sq.pl.spatial_scatter(
        adata[adata.obs["sample"] == slide],
        library_id=slide,
        color="total_counts",
        title=f"{slide} - Total counts",
        size=1.5,
        alpha=1,
        img_alpha=0.2,
        ax=axes[i],
    )

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "total_counts_spatial.pdf"))
plt.show()

In [ ]:
# Plot histogram and density of library sizes
plt.figure(figsize=(8, 6))
sns.histplot(adata.obs['total_counts'], kde=True, color='gray', label='Density')
plt.axvline(x=700, color='red', linestyle='--', label='Threshold (700)')
plt.title('Distribution of Library Sizes')
plt.xlabel('Library Size (Total Counts)')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
# Apply a threshold for low library size
lib_size_threshold = 150
adata.obs['qc_lib_size'] = adata.obs['total_counts'] < lib_size_threshold

# Count the number of spots flagged as low library size
low_library_count = adata.obs['qc_lib_size'].sum()
print(f"Number of spots with low library size: {low_library_count}")

# Make it categorical with readable labels
adata.obs["qc_lib_size_plot"] = (
    adata.obs["qc_lib_size"]
    .map({True: "low", False: "ok"})
    .astype("category")
)

In [ ]:
qc_lib_size_order = sorted(adata.obs["qc_lib_size_plot"].dropna().unique())

# Visualize spots flagged for low library size
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 6 * nrows))
axes = axes.flatten()

for i, slide in enumerate(slides):
    ad = adata[adata.obs["sample"] == slide].copy()

    # re-enforce category order after subsetting
    ad.obs["qc_lib_size_plot"] = ad.obs["qc_lib_size_plot"].cat.set_categories(["low", "ok"])
    
    sq.pl.spatial_scatter(
        ad,
        library_id=slide,
        color=["qc_lib_size_plot"],
        size=1.5,
        title=f"{slide}",
        img_alpha=.5,
        palette=ListedColormap(["blue", "lightgray"]),
        ax=axes[i],
        legend_loc=None
    )

    axes[i].set_axis_off()

handles = [
    Patch(facecolor=cmap(i), label=cat)
    for i, cat in enumerate(qc_lib_size_order)
]

fig.legend(
    handles=handles,
    loc="lower center",
    title="Library size",
    frameon=False
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(os.path.join(OUT_DIR, "total_counts_thresholding.pdf"))
plt.show()

In [ ]:
# fix global category order
histology_order = sorted(adata.obs["histology"].dropna().unique())
adata.obs["histology"] = pd.Categorical(
    adata.obs["histology"],
    categories=histology_order,
    ordered=True
)

# palette in the same fixed order
palette = sns.color_palette("muted", n_colors=len(histology_order))
cmap = ListedColormap(palette)

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 6 * nrows))
axes = axes.flatten()

for i, slide in enumerate(slides):
    ad = adata[adata.obs["sample"] == slide].copy()

    # preserve all categories after subsetting
    ad.obs["histology"] = ad.obs["histology"].cat.set_categories(histology_order)

    sq.pl.spatial_scatter(
        ad,
        library_id=slide,
        color="histology",
        size=1.5,
        title=f"{slide}",
        palette=cmap,
        img_alpha=0.5,
        ax=axes[i],
        legend_loc=None
    )

    axes[i].set_axis_off()

handles = [
    Patch(facecolor=cmap(i), label=cat)
    for i, cat in enumerate(histology_order)
]

fig.legend(
    handles=handles,
    loc="lower center",
    title="Histology",
    frameon=False
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(os.path.join(OUT_DIR, "histology_spatial.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes = axes.flatten()

# fix global color scale
vmin = adata.obs["n_genes_by_counts"].min()
vmax = adata.obs["n_genes_by_counts"].max()

for i, slide in enumerate(slides):
    ad = adata[adata.obs["sample"] == slide]

    sq.pl.spatial_scatter(
        ad,
        library_id=slide,
        color="n_genes_by_counts",
        title=f"{slide}",
        size=1.5,
        alpha=1,
        img_alpha=0.2,
        vmin=vmin,
        vmax=vmax,
        ax=axes[i],
        colorbar=False
    )

    axes[i].set_axis_off()

# for j in range(i + 1, len(axes)):
#     axes[j].axis("off")

import matplotlib as mpl

norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
sm = mpl.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=axes,
    location="right",
    fraction=0.02,
    pad=0.02
)
cbar.set_label("Genes by Counts")

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.savefig(os.path.join(OUT_DIR, "n_genes_spatial.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Plot the distribution of expressed genes
plt.figure(figsize=(8, 6))
sns.histplot(adata.obs['n_genes_by_counts'], kde=True, color='gray')
plt.title('Distribution of Expressed Genes per Spot')
plt.xlabel('Number of Expressed Genes')
plt.ylabel('Density')
plt.axvline(x=100, color='red', linestyle='--', label='Threshold (600)')
plt.legend()
plt.show()

In [ ]:
# Apply the threshold for the number of expressed genes
threshold_expressed_genes = 100
adata.obs['qc_expressed_genes'] = adata.obs['n_genes_by_counts'] < threshold_expressed_genes

# Count the number of spots flagged as low-quality based on expressed genes
low_quality_genes_count = adata.obs['qc_expressed_genes'].sum()
print(f"Number of spots with fewer than {threshold_expressed_genes} expressed genes: {low_quality_genes_count}")

# Make it categorical with readable labels
adata.obs["qc_expressed_genes_plot"] = (
    adata.obs["qc_expressed_genes"]
    .map({True: "low", False: "ok"})
    .astype("category")
)

In [ ]:
qc_genes_order = sorted(adata.obs["qc_expressed_genes_plot"].dropna().unique())

cmap = ListedColormap(["blue", "lightgray"])

# Visualize spots flagged for low library size
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 6 * nrows))
axes = axes.flatten()

for i, slide in enumerate(slides):
    ad = adata[adata.obs["sample"] == slide].copy()

    # re-enforce category order after subsetting
    ad.obs["qc_expressed_genes_plot"] = ad.obs["qc_expressed_genes_plot"].cat.set_categories(["low", "ok"])
    
    sq.pl.spatial_scatter(
        ad,
        library_id=slide,
        color=["qc_expressed_genes_plot"],
        size=1.5,
        title=f"{slide}",
        img_alpha=.5,
        palette=cmap,
        ax=axes[i],
        legend_loc=None
    )

    axes[i].set_axis_off()

handles = [
    Patch(facecolor=cmap(i), label=cat)
    for i, cat in enumerate(qc_genes_order)
]

fig.legend(
    handles=handles,
    loc="lower center",
    title="Genes by Counts",
    frameon=False
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(os.path.join(OUT_DIR, "n_genes_thresholding.pdf"))
plt.show()

In [ ]:
fig = sc.pl.highest_expr_genes(adata, show=False)
fig.figure.savefig(f"{OUT_DIR}/highest_expr_genes.pdf")
plt.show()

### Filtering

In [ ]:
print(f"Number of spots before filtering: {adata.shape[0]}")

# Step 1: Check the number of discarded spots for each metric
metrics_to_check = ['qc_lib_size', 'qc_expressed_genes']
discarded_counts = {}
for metric in metrics_to_check:
    if metric in adata.obs.columns:
        discarded_counts[metric] = adata.obs[metric].sum()
    else:
        print(f"Warning: '{metric}' column not found in adata.obs.")

print("Number of discarded spots for each metric:")
print(discarded_counts)

# Step 2: Combine the set of discarded spots
if all(metric in adata.obs.columns for metric in metrics_to_check):
    adata.obs['discard'] = (
        adata.obs['qc_lib_size'] |
        adata.obs['qc_expressed_genes']
    )
    print(f"Total spots marked as discarded: {adata.obs['discard'].sum()}")
    adata.obs["discard_plot"] = (
        adata.obs["discard"]
        .map({True: "Discard", False: "Keep"})
        .astype("category")
    )
else:
    print("Error: One or more required QC metrics are missing. Cannot combine discarded spots.")

adata.obs["discard_plot"] = adata.obs["discard_plot"].cat.set_categories(["Discard", "Keep"])
qc_discard_order = sorted(adata.obs["discard_plot"].dropna().unique())

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 6 * nrows))
axes = axes.flatten()

# Step 3: Visualize the spatial pattern of combined discarded spots
for i, slide in enumerate(slides):
    ad = adata[adata.obs["sample"] == slide].copy()
    ad.obs["discard_plot"] = ad.obs["discard_plot"].cat.set_categories(["Discard", "Keep"])
    if 'discard' in adata.obs.columns:
        import scanpy as sc
        sq.pl.spatial_scatter(
            ad,
            library_id=slide,
            color='discard_plot',
            size=1.5,
            title=f"{slide}",
            img_alpha=.5,
            ax=axes[i],
            legend_loc=None,
            palette=cmap
        )
    else:
        print("Error: 'discard' column not found in adata.obs. Cannot visualize.")

axes[i].set_axis_off()

handles = [
    Patch(facecolor=cmap(i), label=cat)
    for i, cat in enumerate(qc_discard_order)
]

fig.legend(
    handles=handles,
    loc="lower center",
    title="Discarded Spots",
    frameon=False
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(os.path.join(OUT_DIR, "thresholding.pdf"))
plt.show()

# Step 4: Discard marked cells
adata = adata[adata.obs['discard'] == False].copy()
print(f"Number of spots retained after filtering: {adata.shape[0]}")

Filter out genes detected in less than three spots.

In [ ]:
print(f"Number of genes before filtering: {adata.n_vars}")
sc.pp.filter_genes(adata, min_cells=3)
print(f"Number of genes retained: {adata.n_vars}")

In [ ]:
adata.layers['counts'] = adata.X.copy()

In [ ]:
# anndata_utils.write_10x_h5(adata, os.path.join(OUT_DIR, "filtered_feature_bc_matrix.h5"))
adata.write(os.path.join(OUT_DIR, "adata_filtered_raw.h5ad"))

### Normalization

In [ ]:
adata.layers['library-nomalized'] = sc.pp.normalize_total(adata, target_sum=1e4, exclude_highly_expressed=True, copy=True).X
adata.layers['log-transformed'] = sc.pp.log1p(adata, copy=True).X

In [ ]:
adata.layers['scaled'] = adata.X.copy()
sc.pp.scale(adata, zero_center=True, layer='scaled')
sc.pp.filter_genes(adata.layers['scaled'], min_counts=1)

In [ ]:
original_counts = adata.layers['counts'].sum(axis=1)
normalized_counts = adata.layers['library-nomalized'].sum(axis=1)
scaled_counts = adata.layers['scaled'].sum(axis=1)
log_transformed_counts = adata.layers['log-transformed'].sum(axis=1)

plt.figure(figsize=(8, 6))
sns.histplot(original_counts.A1, color="blue", label="Before normalization", kde=True)
sns.histplot(normalized_counts.A1, color="orange", label="Library-normalized", kde=True)
sns.histplot(scaled_counts, color="green", label="Normalized and scaled", kde=True)
sns.histplot(log_transformed_counts, color="lightblue", label="Log-normalized", kde=True)
plt.legend()
plt.title(f'Effect of Normalization and Scaling')
plt.xlabel('Total Expression')
plt.ylabel('Number of Cells')

In [ ]:
adata.layers['pearson'] = adata.X.copy()
sc.experimental.pp.normalize_pearson_residuals(adata, layer='pearson')

In [ ]:
original_counts = adata.layers['counts'].sum(axis=1)
normalized_counts = adata.layers['pearson'].sum(axis=1)

plt.figure(figsize=(8, 5))
sns.histplot(original_counts.A1, color="blue", label="Before Normalization", kde=True)
sns.histplot(normalized_counts, color="orange", label="After Normalization", kde=True)
plt.xlabel("Total Expression per Spot")
plt.ylabel("Number of Spots")
plt.title("Effect of Pearson Normalization on Expression Distribution")
plt.legend()
plt.savefig(os.path.join(OUT_DIR, "normalization_effect.png"), dpi=600)
plt.show()

In [ ]:
adata.write_h5ad(os.path.join(WORKING_DIR, "adata.h5ad"))